# 06 — RoBERTa (AutoModelForMultipleChoice) for the Smart MCQ Solver

## 1. Setup

Imports + reproducibility seeds + device detection.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForMultipleChoice, get_linear_schedule_with_warmup

In [2]:
# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [3]:
# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

Device: cuda


## 2. Configuration

Key choices i made here:
- **`max_len=128`** — per-option length (prompt + ONE option), not all 5 at once
- **`batch_size=8`** — each example is 5x wider than DeBERTa's per-option batches (5 choices stacked)
- **`lr=2e-5`** — standard transformer fine-tuning LR
- **`epochs=5`** — transformers converge fast on small datasets
- **`warmup_ratio=0.1`** — 10% of steps ramp LR from 0 → max before decaying

In [4]:
CFG = dict(
    model_name   = 'roberta-base',
    max_len      = 128,     # per-option length; prompt + one option, not all 5 at once
    batch_size   = 8,       # 5x wider per example than DeBERTa's per-option batches (5 choices stacked)
    lr           = 2e-5,
    weight_decay = 0.01,
    epochs       = 5,
    warmup_ratio = 0.1,
    patience     = 3,
    seed         = SEED,
)

In [5]:
DATA_DIR = Path('/kaggle/input/competitions/smart-mcq-solver-challenge')
if not DATA_DIR.exists():
    DATA_DIR = Path('data')

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('outputs')
MODEL_DIR  = OUTPUT_DIR / 'models'
PRED_DIR   = OUTPUT_DIR / 'predictions'
for d in (OUTPUT_DIR, MODEL_DIR, PRED_DIR):
    d.mkdir(parents=True, exist_ok=True)

In [6]:
# Answer <-> int maps
ANSWER_MAP  = {'A':0,'B':1,'C':2,'D':3,'E':4}
REVERSE_MAP = {v:k for k,v in ANSWER_MAP.items()}
OPTION_COLS = list('ABCDE')
print(CFG)

{'model_name': 'roberta-base', 'max_len': 128, 'batch_size': 8, 'lr': 2e-05, 'weight_decay': 0.01, 'epochs': 5, 'warmup_ratio': 0.1, 'patience': 3, 'seed': 42}


## 3. Load data + stratified split

In [7]:
def load_raw():
    """Load competition CSVs."""
    train_df = pd.read_csv(DATA_DIR / 'train.csv')
    test_df  = pd.read_csv(DATA_DIR / 'test.csv')
    return train_df, test_df

In [8]:
def lowercase_columns(df, cols=['prompt']+OPTION_COLS):
    """Lowercase + strip whitespace on every text column."""
    df = df.copy()
    for col in cols:
        df[col] = df[col].astype(str).str.lower().str.strip()
    return df

In [9]:
def stratified_split(train_df, val_size=0.2, seed=SEED):
    """Manual stratified split on the 'answer' column."""
    np.random.seed(seed)
    train_idx, val_idx = [], []
    for ans in 'ABCDE':
        idx = train_df[train_df['answer']==ans].index.tolist()
        np.random.shuffle(idx)
        cut = int(len(idx)*(1.0-val_size))
        train_idx += idx[:cut]
        val_idx   += idx[cut:]
    tr = train_df.loc[train_idx].reset_index(drop=True)
    va = train_df.loc[val_idx].reset_index(drop=True)
    return tr, va

In [10]:
# Load + clean
train_raw, test_df = load_raw()
train_raw = lowercase_columns(train_raw)
test_df   = lowercase_columns(test_df)

# Stratified 80/20
tr_df, va_df = stratified_split(train_raw)
print(f'train: {len(tr_df)} | val: {len(va_df)} | test: {len(test_df)}')

train: 1599 | val: 401 | test: 500


## 4. Dataset — the multiple-choice input format

`AutoModelForMultipleChoice` expects, per example, a **stack of 5 encodings**
(one per option) each shaped `(seq_len,)`, giving a batch tensor of shape
`(batch, 5, seq_len)`.

Each of the 5 encodings is `(prompt, option_text)` tokenized as a pair —
the model sees the full question once per option, and produces one logit
per option, then a joint softmax over the 5.

In [11]:
class MCQDatasetHF(Dataset):
    """Dataset for AutoModelForMultipleChoice — returns 5 stacked encodings per example."""
    def __init__(self, df, tokenizer, max_len, is_test=False):
        self.df = df.reset_index(drop=True)
        self.tok = tokenizer
        self.max_len = max_len
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt  = row['prompt']
        options = [row[c] for c in OPTION_COLS]

        # Tokenize (prompt, option) as a pair, once per option
        enc = self.tok(
            [prompt]*5, options,
            max_length=self.max_len, truncation=True, padding='max_length',
            return_tensors='pt',
        )
        item = {
            'input_ids':      enc['input_ids'],       # (5, max_len)
            'attention_mask': enc['attention_mask'],   # (5, max_len)
        }
        if not self.is_test:
            item['labels'] = torch.tensor(ANSWER_MAP[row['answer']], dtype=torch.long)
        return item

In [12]:
# Load tokenizer + build datasets/loaders
tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'])

tr_ds = MCQDatasetHF(tr_df, tokenizer, CFG['max_len'], is_test=False)
va_ds = MCQDatasetHF(va_df, tokenizer, CFG['max_len'], is_test=False)
te_ds = MCQDatasetHF(test_df, tokenizer, CFG['max_len'], is_test=True)

tr_loader = DataLoader(tr_ds, batch_size=CFG['batch_size'], shuffle=True,  num_workers=0, pin_memory=True)
va_loader = DataLoader(va_ds, batch_size=CFG['batch_size'], shuffle=False, num_workers=0, pin_memory=True)
te_loader = DataLoader(te_ds, batch_size=CFG['batch_size'], shuffle=False, num_workers=0, pin_memory=True)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [13]:
# Quick check: pull one batch and inspect shapes
batch = next(iter(tr_loader))
print('input_ids shape :', batch['input_ids'].shape)   # (batch, 5, max_len)
print('labels shape    :', batch['labels'].shape)

input_ids shape : torch.Size([8, 5, 128])
labels shape    : torch.Size([8])


## 5. Model

`AutoModelForMultipleChoice` already includes the multiple-choice
classification head (a linear layer producing one logit per choice from the
pooled `[CLS]` representation) and computes cross-entropy loss internally
when `labels` are passed — no need to write a custom head or loss, unlike
the DeBERTa notebook's custom `DeBERTaOptionScorer`.


In [14]:
# Load pretrained RoBERTa with the multiple-choice head
model = AutoModelForMultipleChoice.from_pretrained(CFG['model_name']).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f'{CFG["model_name"]} loaded — {total_params/1e6:.1f}M params')

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForMultipleChoice LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.pooler.dense.weight     | MISSING    | 
classifier.bias                 | MISSING    | 
roberta.pooler.dense.bias       | MISSING    | 
classifier.weight               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


roberta-base loaded — 124.6M params


## 6. MAP@3 metric

Same MAP@3 formula used across the project:

| Position of correct answer | Score |
|----------------------------|-------|
| 1st                        | 1.00  |
| 2nd                        | 0.50  |
| 3rd                        | 0.33  |
| Not in top-3               | 0.00  |

In [15]:
def map_at_3_from_logits(logits, labels):
    """Compute MAP@3 from raw logits and integer labels."""
    probs  = torch.softmax(logits, dim=1).cpu().numpy()
    labels = labels.cpu().numpy()
    scores = []
    for i, true in enumerate(labels):
        top3 = np.argsort(probs[i])[-3:][::-1]
        hit  = np.where(top3 == true)[0]
        scores.append(1.0/(hit[0]+1) if len(hit) else 0.0)
    return float(np.mean(scores))

## 7. WandB setup

In [16]:
import wandb
try:
    from kaggle_secrets import UserSecretsClient
    key = UserSecretsClient().get_secret('WANDB_API_KEY')
    wandb.login(key=key)
    USE_WANDB = True
    print('[OK] WandB logged in!')
except Exception:
    USE_WANDB = False
    print('WandB secret not found — continuing without it.')

if USE_WANDB:
    run = wandb.init(
        project='23f2003236-t22026',
        name='roberta_mc_v1',
        config=CFG,
        tags=['roberta','pretrained','transformer','multiple-choice','day9'],
        reinit=True
    )
    print(f'Run URL: {run.url}')

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 23f2003236 (23f2003236-iit-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


[OK] WandB logged in!


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: setting up run hkrg0qxz
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260713_162941-hkrg0qxz
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run roberta_mc_v1
wandb: ⭐️ View project at https://wandb.ai/23f2003236-iit-madras/23f2003236-t22026
wandb: 🚀 View run at https://wandb.ai/23f2003236-iit-madras/23f2003236-t22026/runs/hkrg0qxz


Run URL: https://wandb.ai/23f2003236-iit-madras/23f2003236-t22026/runs/hkrg0qxz
